<a href="https://colab.research.google.com/github/aryankr8121/MiniGPT/blob/main/MiniGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import requests
import os

# ----------------------------------------------------------------------------
# HYPERPARAMETERS
# These control model size, training length, and behavior. The defaults are
# small so it trains in a few minutes on CPU. Comments show what to raise if
# you have a GPU and want better output.
# ----------------------------------------------------------------------------
batch_size   = 32     # how many independent sequences we process in parallel
block_size    = 128    # context length: max tokens the model sees to predict the next one
max_iters     = 3000   # total training steps
eval_interval = 300    # how often to print train/val loss
eval_iters    = 200    # how many batches to average when estimating loss
learning_rate = 3e-4   # step size for the optimizer
n_embd        = 192    # embedding dimension (width of the model)
n_head        = 6      # number of attention heads (n_embd must divide by this)
n_layer       = 6      # number of transformer blocks stacked
dropout       = 0.2    # fraction of activations randomly zeroed (regularization)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)  # reproducibility

# ----------------------------------------------------------------------------
# TOKENIZER (character-level)
# Build a vocabulary of every unique character, then map chars <-> integers.
# ----------------------------------------------------------------------------

# Ensure input.txt is present
file_name = 'input.txt'
if not os.path.exists(file_name):
    print(f"File '{file_name}' not found. Attempting to download...")
    # Correct URL for the tiny shakespeare dataset
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    try:
        response = requests.get(url)
        response.raise_for_status() # Raise an exception for bad status codes
        with open(file_name, 'w', encoding='utf-8') as f:
            f.write(response.text)
        print(f"Downloaded {file_name} successfully within cell K9_qOrcUHDrT.")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading {file_name}: {e}")
        raise FileNotFoundError(f"Failed to download {file_name}. Original error: {e}")

with open(file_name, 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))     # all unique characters, sorted
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}   # string -> integer
itos = {i: ch for i, ch in enumerate(chars)}   # integer -> string
encode = lambda s: [stoi[c] for c in s]                 # text  -> list of ints
decode = lambda l: ''.join([itos[i] for i in l])        # ints  -> text

# ----------------------------------------------------------------------------
# DATA: encode the whole text, split 90% train / 10% validation
# ----------------------------------------------------------------------------
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    """Grab a random batch of input (x) and target (y) chunks.
    y is x shifted by one position: at every spot, predict the next char."""
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))      # random start points
    x = torch.stack([d[i:i + block_size] for i in ix])           # (batch, block_size)
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])   # same, shifted by 1
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss():
    """Average loss over several batches for a stable readout (no gradients)."""
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# ----------------------------------------------------------------------------
# SELF-ATTENTION: one head
# Each token emits a query (what am I looking for?), a key (what do I offer?),
# and a value (what do I pass on?). Attention weight = how well a query matches
# each key. A causal mask blocks attention to future positions.
# ----------------------------------------------------------------------------
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # lower-triangular matrix used to mask out the future (not a learned param)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)     # (B, T, head_size)
        q = self.query(x)   # (B, T, head_size)
        # attention scores: how much each token attends to every other token
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5    # scaled dot-product
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # causal mask
        wei = F.softmax(wei, dim=-1)    # turn scores into probabilities
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v      # weighted sum of values -> (B, T, head_size)

class MultiHeadAttention(nn.Module):
    """Several attention heads in parallel; their outputs are concatenated."""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)  # mix the heads back together
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    """A small MLP applied per-position. Lets the model 'think' on what
    attention gathered. Expands to 4x width then back, standard in transformers."""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """One transformer block: attention + feed-forward, each wrapped with a
    residual connection (x + ...) and layer norm for stable training."""
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))     # attention, with residual
        x = x + self.ffwd(self.ln2(x))   # feed-forward, with residual
        return x

# ----------------------------------------------------------------------------
# THE FULL GPT MODEL
# ----------------------------------------------------------------------------
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # each token id -> a learned vector
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # each position (0..block_size-1) -> a learned vector (gives order info)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)               # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)   # project to vocab logits

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                          # (B,T,n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (T,n_embd)
        x = tok_emb + pos_emb       # combine token identity + position
        x = self.blocks(x)          # run through the transformer stack
        x = self.ln_f(x)
        logits = self.lm_head(x)    # (B, T, vocab_size) -> score per next-token

        if targets is None:
            loss = None
        else:
            # reshape so cross-entropy compares predicted vs actual next char
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        """Generate text one token at a time."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]        # crop to last block_size tokens
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature  # focus on last position
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # sample a token
            idx = torch.cat((idx, idx_next), dim=1)             # append it
        return idx

# ----------------------------------------------------------------------------
# TRAIN
# ----------------------------------------------------------------------------
model = GPTLanguageModel().to(device)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for it in range(max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        losses = estimate_loss()
        print(f"step {it:4d}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)   # clear old gradients
    loss.backward()                          # backpropagate
    optimizer.step()                         # update weights

# ----------------------------------------------------------------------------
# GENERATE a sample after training
# ----------------------------------------------------------------------------
print("\n--- Generated sample ---")
context = torch.zeros((1, 1), dtype=torch.long, device=device)  # start token (newline)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))

2.72M parameters
step    0: train loss 4.3581, val loss 4.3513
step  300: train loss 2.4057, val loss 2.4242
step  600: train loss 2.1404, val loss 2.1799
step  900: train loss 1.9663, val loss 2.0373
step 1200: train loss 1.8365, val loss 1.9481
step 1500: train loss 1.7449, val loss 1.8842
step 1800: train loss 1.6745, val loss 1.8346
step 2100: train loss 1.6184, val loss 1.7939
step 2400: train loss 1.5782, val loss 1.7600
step 2700: train loss 1.5437, val loss 1.7294
step 2999: train loss 1.5127, val loss 1.7007

--- Generated sample ---

Near namenty, soe a parrios
With negge of me to give not you. Warwick'd how,
Peasred nael but and my montrent any the know!

BAHY BRuling:
My hopsenly morries, Say
To loss the faithine to in.
'Tiwes need, were laming discelftion to
Yet maning of that deatish this lefMence again:
Let his earth, father-mistrest to offels at
Ber bose the way, now come should tonguanth of this.

JULIET:
More in grear your honousang: I much to firluck there,
Foot of l